# SuperMimic on Colab

One-time setup:
- Runtime, change runtime type, pick a GPU.
- Colab Secrets (key icon on the left): add `WANDB_API_KEY` and allow notebook access.

Run the cells top to bottom. Everything lives in `/content` and is rebuilt every session: the Python environment and the motion files. Checkpoints are uploaded to W&B by the training run, so nothing needs Google Drive.

## 1. GPU check

In [ ]:
import subprocess
out = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print(out)
assert out, "No GPU found. Change the runtime type to a GPU."
driver = out.splitlines()[0].split(",")[1].strip()
assert int(driver.split(".")[0]) >= 580, (
    f"NVIDIA driver {driver} is too old. The pinned torch build (CUDA 13) needs driver 580 or newer."
)

## 2. Repos

In [ ]:
# No Google Drive: motions are rebuilt each session and checkpoints go to W&B.
STORE = "/content/store"
WS = "/content/supermimic_ws"

# Sibling clones: pyproject.toml resolves mjlab from ../mjlab_research.
!mkdir -p {WS} {STORE}/data
!if [ -d {WS}/mjlab_research ]; then git -C {WS}/mjlab_research pull --ff-only; else git clone --branch main https://github.com/xSurus/mjlab_research.git {WS}/mjlab_research; fi
!if [ -d {WS}/supermimic ]; then git -C {WS}/supermimic pull --ff-only; else git clone --branch thesis https://github.com/xSurus/supermimic.git {WS}/supermimic; fi
# data/ is not in .gitignore, so it stays outside the repo and cells use absolute paths.

## 3. Environment

In [ ]:
!pip install -q uv
!cd {WS}/supermimic && env -u PYTHONPATH uv sync --locked --group dev
out = !cd {WS}/supermimic && env -u PYTHONPATH uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
print("\n".join(out))
assert out and out[-1].endswith("True"), "The uv environment is missing or torch cannot see the GPU. Read the uv sync output above."

## 4. Weights & Biases

In [ ]:
import os
from google.colab import userdata

try:
    key = userdata.get("WANDB_API_KEY")
except Exception:
    key = None
assert key, "Add your W&B API key to Colab Secrets as WANDB_API_KEY and allow notebook access"
os.environ["WANDB_API_KEY"] = key

## 5. Tests

In [ ]:
!cd {WS}/supermimic && MUJOCO_GL=disable env -u PYTHONPATH uv run pytest -q -rs

## 6. Motions

Downloads the LAFAN1 G1 retargeted CSVs and converts them to npz under `/content/store`. Existing files are skipped.

In [ ]:
!cd {WS}/supermimic && env -u PYTHONPATH uv run --with huggingface_hub hf download lvhaidong/LAFAN1_Retargeting_Dataset --repo-type dataset --include "g1/*" --local-dir {STORE}/data/raw/lafan1
!cd {WS}/supermimic && env -u PYTHONPATH uv run python colab/build_motions.py --csv-dir {STORE}/data/raw/lafan1/g1 --out-dir {STORE}/data/npz/lafan1

## 7. Smoke run

In [ ]:
# The first update compiles with torch.compile.
# On Colab's 2 CPU cores this can print nothing for many minutes.
# That is expected. Wait for it.
# For a quick check only, add TORCHDYNAMO_DISABLE=1 before "env" to skip compilation.
# Updates are then slower.
!cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 --motion-file {STORE}/data/npz/lafan1/walk1_subject1.npz --num-envs 256 --runner.max-iterations 300 --runner.run-name colab-smoke

## 8. Single-clip training run

One clip, 2048 robots, 5000 loop steps. This gives a first trained policy and real timings on this GPU.

The policy batch is 1024, not the default 4096. The policy update backpropagates through 32 imagined steps for every rollout in the batch, and at 4096 that needs more than the T4's 15 GB (it failed at 2048 robots with the default batch). If it still runs out of memory, set `--runner.policy.batch-size 512`.

In [ ]:
# Checkpoints upload to W&B every 1000 steps. The first update compiles and can print nothing for many minutes.
!cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 --motion-file {STORE}/data/npz/lafan1/walk1_subject1.npz --num-envs 2048 --runner.policy.batch-size 1024 --runner.max-iterations 5000 --runner.run-name walk1-2048-pb1024

## 9. Full run

In [ ]:
# !cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 --motion-file '{STORE}/data/npz/lafan1/*.npz' --num-envs 4096

## 10. Play

In [ ]:
# A 300-iteration smoke run only saves the step-0 checkpoint, so play a longer run.
# Replace the placeholders with the W&B entity and run id.
# !cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/play.py SuperMimic-GenCoord-G1 --wandb-path <entity>/supermimic/<run id>